In [1]:
import matplotlib.pyplot as plt
from netCDF4 import Dataset

import sys

import MaNTA

from FFIRunner import FFIRunner

Using gpu implementation


In [2]:
import jax

import jax.numpy as jnp

import jax.scipy.special as sci
import jax.scipy.integrate as integrate

from functools import partial

from jax.flatten_util import ravel_pytree

vmap_axes_adj = (None, {"Variable": 0, "Derivative": 0, "Flux": 0, "Aux": 0, "Scalars": None}, 0, None,  0)

class JAXAdjointProblem(MaNTA.AdjointProblem):
    def __init__(self, transport_system: MaNTA.TransportSystem, g):
        MaNTA.AdjointProblem.__init__(self)
        self.params = transport_system.params
        self.g = g


        self.ng = 1
        self.np = len(transport_system.params)
        # print(self.np_cell)
        self.npoints = len(transport_system.points)
        # self.np = self.np_cell * self.npoints
        self.np_boundary = 0
        self.spatialParameters = True
        self.sigma = transport_system.sigma
        self.source = transport_system.source

        self.daux_dp = jax.jit(jax.grad(transport_system.aux, argnums=4))

        self.UpperBoundarySensitivities = {}
        self.LowerBoundarySensitivities = {}

    def setParams(self, params):
        self.params = params

    def gFn(self, i, states, positions):
        x = jnp.array(positions)
        out =  jax.vmap(self.g, in_axes=({"Variable": 0, "Derivative": 0, "Flux": 0, "Aux": 0, "Scalars": None}, 0, 0))(states, x, self.params)
        return out

    #@partial(jax.jit, static_argnums=(0,1))
    def dgFndp(self, gIndex, states, positions):
        x = jnp.array(positions)
        dgdp = jax.vmap(jax.grad(self.g, argnums=2), in_axes=({"Variable": 0, "Derivative": 0, "Flux": 0, "Aux": 0, "Scalars": None}, 0, 0))(states, x, self.params)
        g, _ = ravel_pytree(dgdp)
        g = jnp.reshape(g, (len(positions),self.np - self.np_boundary))

        out = jnp.pad(g, pad_width=(self.np_boundary, 0), mode='constant', constant_values=0)
        print(out.shape)
        return out
        

    @partial(jax.jit, static_argnums=(0,))
    def dg(self, i, states, positions):
        x = jnp.array(positions)

        out = jax.vmap(jax.grad(self.g, argnums=0), in_axes=({"Variable": 0, "Derivative": 0, "Flux": 0, "Aux": 0, "Scalars": None}, 0, None))(states, x, self.params)  
        out["Scalars"] = []
        return out

    #@partial(jax.jit, static_argnums=(0,))
    def dSigma(self, i, states, positions):
        x = jnp.array(positions)

        grad = jax.vmap(jax.grad(self.sigma, argnums=4), in_axes=(vmap_axes_adj))(i, states, x, 0.0, self.params)  
        
        grad_flattened, _ = jax.flatten_util.ravel_pytree(grad)
        grad_flattened = jnp.expand_dims(grad_flattened, 1)
        out = jnp.reshape(grad_flattened, (self.np , self.npoints ))
        return out
    
    
   # @partial(jax.jit, static_argnums=(0,))
    def dSources(self, i, states, positions):
        x = jnp.array(positions)
        grad = jax.vmap(jax.grad(self.source, argnums=4), in_axes=(vmap_axes_adj))(i, states, x, 0.0, self.params)  
        grad_flattened, _ = jax.flatten_util.ravel_pytree(grad)
        grad_flattened = jnp.expand_dims(grad_flattened, 1)
        out = jnp.reshape(grad_flattened, (self.np, self.npoints ))
        return out

    @partial(jax.jit, static_argnums=(0,))
    def dgFn_dphi(self, i, state, x):
        return jax.grad(self.g, argnums=0)(state, x, self.params)["Aux"]
   
    def dAux_dp(self, index, pIndex, state, x):
        return self.daux_dp(index, state, x, 0.0, self.params )[pIndex]
    
    def computeUpperBoundarySensitivity(self, i, pIndex):
        if (i, pIndex) in self.UpperBoundarySensitivities:
            return True
        else:
            return False
        
    def computeLowerBoundarySensitivity(self, i, pIndex):
        if (i, pIndex) in self.LowerBoundarySensitivities:
            return True
        else:
            return False
    
    def getName(self, pIndex):
        if pIndex < len(self.params):
            return list(self.params._fields)[pIndex]
        else:
            return "BoundaryCondition"+str(pIndex)
        
    def addUpperBoundarySensitivity(self, i):
        self.UpperBoundarySensitivities[(i,self.np)] = True
        self.np += 1
        self.np_boundary += 1

    def addLowerBoundarySensitivity(self, i):
        self.LowerBoundarySensitivities[(i,self.np)] = True
        self.np += 1
        self.np_boundary += 1
    
   

In [3]:
from typing import NamedTuple
from VectorizedTransportSystem import VectorizedTransportSystem

vmap_axes = ({"Variable": 0, "Derivative": 0, "Flux": 0, "Aux": 0, "Scalars": None}, 0, 0)

class JAXNonlinearDiffusion(VectorizedTransportSystem):
    def __init__(self, kappa):
        super().__init__()
        self.nVars = 1
        self.isUpperDirichlet  = True
        self.isLowerDirichlet  = False
        
        self.T_s = 50.0
        self.SourceWidth = 0.02
        self.SourceCentre = 0.3
        self.a = 0.0

        solver_config = {
            "OutputFilename": "out",
            "Polynomial_degree": 10,
            "Grid_size": 10,
            "tau": 1.0, 
            "Lower_boundary": 0.0,
            "Upper_boundary": 1.0,
            "Relative_tolerance": 0.01,
            "delta_t": 1.0,
            "restart": False,
            "solveAdjoint": True, 
        }
        self.kappa = kappa
        self.points = MaNTA.getNodes(solver_config["Lower_boundary"], solver_config["Upper_boundary"], solver_config["Grid_size"], solver_config["Polynomial_degree"])

        self.D = lambda x, kappa : (2* x + 1)* kappa

        self.params = {
            "D": self.D(self.points, self.kappa),
            "T_s": self.T_s * jnp.ones(len(self.points)),
            "a": self.a * jnp.ones(len(self.points)),
            "SourceWidth": self.SourceWidth * jnp.ones(len(self.points)),
            "SourceCentre": self.SourceCentre * jnp.ones(len(self.points))
        }
        self.adjointProblem = JAXAdjointProblem(self, self.g)
        self.runner = FFIRunner(self, self.points, self.adjointProblem.ng , self.adjointProblem.np, spatialParameters=True)

        self.runner.configure(solver_config)

        # This object will be passed to sigma and source functions
    
    def run(self, tFinal = None):
        
        if (tFinal is not None):
            sFinal = self.runner.run(tFinal)
        else: 
            sFinal = self.runner.run_ss()

        return sFinal

    def runAdjointSolve(self):
        G, G_p = self.runner.run_adjoint_solve()
        return G, G_p

    def g(self, state, x, params):
        u = state["Variable"][0]
        return 0.5 * u * u 

         
    def SigmaFn( self, index, state, x, t ):
        u = state["Variable"][0]
        q = state["Derivative"][0]
        return self.D(x, self.kappa)*(u ** self.a) * q

    def Sources(self, index, state, x, t):
        y = x - self.SourceCentre
        return self.T_s*jnp.exp(-y*y/self.SourceWidth)

    def SigmaFn_v( self, index, states, positions, t):
        x = jnp.array(positions)
        return jax.vmap(lambda s, p, params : self.sigma(index, s, p, t, params), in_axes=(vmap_axes))(states, x, self.params)

    #@partial(jax.jit, static_argnums=(0,))
    def Sources_v( self, index, states, positions, t ):
        x = jnp.array(positions)
        return jax.vmap(lambda s, p, params : self.source(index, s, p, t, params), in_axes=(vmap_axes))(states, x, self.params)
        
    #@partial(jax.jit, static_argnums=(0,))
    def dSigma(self, index, states, positions, t):
        x = jnp.array(positions)
        out =  jax.vmap(lambda s, p, params: jax.grad(self.sigma, argnums=1)(index, s, p, t, params), in_axes=(vmap_axes))(states, x, self.params)
        out["Scalars"] = []

        return out
    
    #@partial(jax.jit, static_argnums=(0,))
    def dSources(self, index, states, positions, t):
        x = jnp.array(positions)
        out =  jax.vmap(lambda s, p, params: jax.grad(self.source, argnums=1)(index, s, p, t, params), in_axes=(vmap_axes))(states, x, self.params)
        out["Scalars"] = []
        return out
    
    def sigma( self, index, state, x, t, params ):
        
        u = state["Variable"][0]
        q = state["Derivative"][0]
        return params["D"]*(u ** params["a"]) * q

    def source( self, index, state, x, t, params ):
        y = x - params["SourceCentre"]
        return params["T_s"]*jnp.exp(-y*y/params["SourceWidth"])


    def LowerBoundary(self, index, t):
        return 0.0

    def UpperBoundary(self, index, t):
        return 0.3
    
    def InitialValue(self, index, x):
        return 0.3
    
    def createAdjointProblem(self):
        return self.adjointProblem

In [ ]:
nl = JAXNonlinearDiffusion(2.0)
nl.run(tFinal = 5.0)

1
550


INFO: Using default value for configuration option High_Grid_Boundary
INFO: Using default value for configuration option Lower_Boundary_Fraction
INFO: Using default value for configuration option Upper_Boundary_Fraction
INFO: Creating grid with 10 cells from x = 0 to x = 1
Total HDG degrees of freedom 341
INFO: Using default value for configuration option Absolute_tolerance
INFO: Using default value for configuration option tZero
INFO: Using default value for configuration option MinStepSize
INFO: Using default value for configuration option OutputPoints
INFO: Using default value for configuration option SteadyStateTolerance
INFO: Using default value for configuration option WriteOutput
Configuration done.
Setting initial conditions


Number of Residual Evaluations due to IDACalcIC 2
Writing output at 1 ( 41 timesteps )
Writing output at 2 ( 43 timesteps )
Writing output at 3 ( 44 timesteps )
Writing output at 4 ( 44 timesteps )
Writing output at 5 ( 44 timesteps )
Total Number of Timesteps             :44
Total Number of Residual Evaluations  :71
Total Number of Jacobian Computations :18
Computing adjoints


INFO: Computing adjoints for 5 parameters


In [ ]:
import numpy as np
from netCDF4 import Dataset

data = Dataset("./out.nc")
print(data)

Vars = data.groups
t = jnp.array(np.array(data.variables["t"]))
x = jnp.array(np.array(data.variables["x"]))
u = jnp.array(np.array(Vars["Var0"].variables["u"]))
data.close()

In [ ]:
fig,ax = plt.subplots()
ax.plot(x,u[-1,:])

In [ ]:
import interpax

G, G_p = nl.runAdjointSolve()

print(G)

u_interp = interpax.interp1d(nl.points, x, u[-1,:], method='cubic')

g_approx = lambda u : jnp.trapezoid(0.5 * u * u, nl.points)

print(g_approx(u_interp))


In [ ]:

print(G_p)
G_p_int = jnp.sum(G_p, axis=0)
print(G_p_int)
plt.plot(nl.points, G_p[:,0])